In [89]:
import polars as pl

pl.Config.set_tbl_rows(24)
pl.Config.save()  # saves to ~/.config/polars/config.json

'{"environment":{"POLARS_ENGINE_AFFINITY":null,"POLARS_FMT_MAX_COLS":null,"POLARS_FMT_MAX_ROWS":"24","POLARS_FMT_NUM_DECIMAL":null,"POLARS_FMT_NUM_GROUP_SEPARATOR":null,"POLARS_FMT_NUM_LEN":null,"POLARS_FMT_STR_LEN":null,"POLARS_FMT_TABLE_CELL_ALIGNMENT":null,"POLARS_FMT_TABLE_CELL_LIST_LEN":null,"POLARS_FMT_TABLE_CELL_NUMERIC_ALIGNMENT":null,"POLARS_FMT_TABLE_DATAFRAME_SHAPE_BELOW":null,"POLARS_FMT_TABLE_FORMATTING":null,"POLARS_FMT_TABLE_HIDE_COLUMN_DATA_TYPES":null,"POLARS_FMT_TABLE_HIDE_COLUMN_NAMES":null,"POLARS_FMT_TABLE_HIDE_COLUMN_SEPARATOR":null,"POLARS_FMT_TABLE_HIDE_DATAFRAME_SHAPE_INFORMATION":null,"POLARS_FMT_TABLE_INLINE_COLUMN_DATA_TYPE":null,"POLARS_FMT_TABLE_ROUNDED_CORNERS":null,"POLARS_MAX_EXPR_DEPTH":null,"POLARS_STREAMING_CHUNK_SIZE":null,"POLARS_TABLE_WIDTH":null,"POLARS_VERBOSE":null,"POLARS_WARN_UNSTABLE":null},"direct":{"set_fmt_float":"mixed","set_float_precision":null,"set_thousands_separator":"","set_decimal_separator":".","set_trim_decimal_zeros":false}}'

In [90]:
lf = pl.scan_parquet(
    "../data/raw/yellow_tripdata_2025-01.parquet"
)  # lazy scan - loads on collect()
df = lf.collect()
df.columns = [c.lower() for c in df.columns]
df = df.rename(
    {
        "tpep_pickup_datetime": "pickup_datetime",
        "tpep_dropoff_datetime": "dropoff_datetime",
        "ratecodeid": "rate_code_id",
        "vendorid": "vendor_id",
        "pulocationid": "pickup_location_id",
        "dolocationid": "dropoff_location_id",
    }
)

In [91]:
# Load and join zones data
zones = pl.read_csv("../data/raw/taxi_zone_lookup.csv")
zones.columns = [c.lower() for c in zones.columns]
df = df.join(
    zones.rename({"locationid": "pickup_location_id"}),
    on="pickup_location_id",
    how="left",
)

In [92]:
zones.filter(pl.col("zone").str.contains("Airport"))

locationid,borough,zone,service_zone
i64,str,str,str
1,"""EWR""","""Newark Airport""","""EWR"""
132,"""Queens""","""JFK Airport""","""Airports"""
138,"""Queens""","""LaGuardia Airport""","""Airports"""


In [ ]:
# build is_airport_pickup field
df = df.with_columns(
    pl.col("pickup_location_id").is_in([1, 132, 138]).cast(pl.Int8).alias("is_airport_pickup")
)

In [94]:
# Build model columns
store_and_fwd_bin = {
    "N": 0,
    "Y": 1,
}

df = df.with_columns(
    # Create duration_sec column from dropoff and pickup times
    ((pl.col("dropoff_datetime") - pl.col("pickup_datetime")).dt.total_seconds() / 60).alias(
        "duration_min"
    ),
    # Convert Y/N flagging to 1-hot encoding
    (
        pl.col("store_and_fwd_flag")
        .replace_strict(store_and_fwd_bin, default=None)
        .alias("store_and_fwd_flag")
    ),
)

In [95]:
# %%script echo skipping
# print("This code will be completely ignored.")
# EDA ONLY — drop before modeling
# Build readability columns
vendor_id_map = {
    1: "Creative Mobile Technologies LLC",
    2: "Curb Mobility, LLC",
    6: "Myle Technologies Inc",
    7: "Helix",
}

payment_type_map = {
    0: "flex_fare_trip",
    1: "credit_card",
    2: "cash",
    3: "no_charge",
    4: "dispute",
    5: "unknown",
    6: "voided_trip",
}


rate_code_map = {
    1: "standard_rate",
    2: "jfk",
    3: "newark",
    4: "nassau_or_winchester",
    5: "negotiated_fare",
    6: "group_ride",
    99: "null_or_unknown",
}

df = df.with_columns(
    # Create a new column with readable payment type
    (
        pl.col("payment_type")
        .replace_strict(payment_type_map, default=None)
        .alias("payment_type_str")
    ),
    # Create a new column with readable rate code
    (pl.col("rate_code_id").replace_strict(rate_code_map, default=None).alias("rate_code_str")),
    # Create a new column with readable vendor ID
    (pl.col("vendor_id").replace_strict(vendor_id_map, default=None).alias("vendor_id_str")),
)

In [96]:
# Add avg mph field
df = df.with_columns(
    (pl.col("trip_distance") / (pl.col("duration_min") / 60)).alias("avg_speed_mph")
)

In [97]:
df.schema

Schema([('vendor_id', Int32),
        ('pickup_datetime', Datetime(time_unit='us', time_zone=None)),
        ('dropoff_datetime', Datetime(time_unit='us', time_zone=None)),
        ('passenger_count', Int64),
        ('trip_distance', Float64),
        ('rate_code_id', Int64),
        ('store_and_fwd_flag', Int64),
        ('pickup_location_id', Int32),
        ('dropoff_location_id', Int32),
        ('payment_type', Int64),
        ('fare_amount', Float64),
        ('extra', Float64),
        ('mta_tax', Float64),
        ('tip_amount', Float64),
        ('tolls_amount', Float64),
        ('improvement_surcharge', Float64),
        ('total_amount', Float64),
        ('congestion_surcharge', Float64),
        ('airport_fee', Float64),
        ('cbd_congestion_fee', Float64),
        ('borough', String),
        ('zone', String),
        ('service_zone', String),
        ('is_airport_pickup', Int8),
        ('duration_min', Float64),
        ('payment_type_str', String),
        ('rate_

In [98]:
df.describe()

statistic,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,borough,zone,service_zone,is_airport_pickup,duration_min,payment_type_str,rate_code_str,vendor_id_str,avg_speed_mph
str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,f64,f64,str,str,str,f64
"""count""",3.475226e6,"""3475226""","""3475226""",2.935077e6,3.475226e6,2.935077e6,2.935077e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,3.475226e6,2.935077e6,2.935077e6,3.475226e6,"""3475226""","""3475226""","""3475226""",3.475226e6,3.475226e6,"""3475226""","""2935077""","""3475226""",3.475226e6
"""null_count""",0.0,"""0""","""0""",540149.0,0.0,540149.0,540149.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,540149.0,540149.0,0.0,"""0""","""0""","""0""",0.0,0.0,"""0""","""540149""","""0""",0.0
"""mean""",1.785428,"""2025-01-17 11:02:55.910964""","""2025-01-17 11:17:56.997901""",1.297859,5.855126,2.482535,0.002605,165.191576,164.125177,1.036623,17.081803,1.317737,0.478099,2.959813,0.449308,0.954795,25.611292,2.225237,0.123911,0.483409,null,null,null,0.067959,15.018116,null,null,null,NaN
"""std""",0.426328,null,null,0.75075,564.6016,11.632772,0.050973,64.529483,69.401686,0.701333,463.472918,1.861509,0.137462,3.779681,2.002582,0.278194,463.658478,0.903993,0.472509,0.361931,null,null,null,0.251675,38.713582,null,null,null,NaN
"""min""",1.0,"""2024-12-31 20:47:55""","""2024-12-18 07:52:40""",0.0,0.0,1.0,0.0,1.0,1.0,0.0,-900.0,-7.5,-0.5,-86.0,-126.94,-1.0,-901.0,-2.5,-1.75,-0.75,"""Bronx""","""Allerton/Pelham Gardens""","""Airports""",0.0,-51472.316667,"""cash""","""group_ride""","""Creative Mobile Technologies L…",-64368.0
"""25%""",2.0,"""2025-01-10 07:59:01""","""2025-01-10 08:15:29""",1.0,0.98,1.0,0.0,132.0,113.0,1.0,8.6,0.0,0.5,0.0,0.0,1.0,15.2,2.5,0.0,0.0,null,null,null,0.0,7.283333,null,null,null,7.2103
"""50%""",2.0,"""2025-01-17 15:41:34""","""2025-01-17 15:59:34""",1.0,1.67,1.0,0.0,162.0,162.0,1.0,12.11,0.0,0.5,2.45,0.0,1.0,19.95,2.5,0.0,0.75,null,null,null,0.0,11.7,null,null,null,9.520343
"""75%""",2.0,"""2025-01-24 19:34:06""","""2025-01-24 19:48:31""",1.0,3.1,1.0,0.0,234.0,234.0,1.0,19.5,2.5,0.5,3.93,0.0,1.0,27.78,2.5,0.0,0.75,null,null,null,0.0,18.333333,null,null,null,12.941176
"""max""",7.0,"""2025-02-01 00:00:44""","""2025-02-01 23:44:11""",9.0,276423.57,99.0,1.0,265.0,265.0,5.0,863372.12,15.0,10.5,400.0,170.94,1.0,863380.37,2.5,6.75,0.75,"""Unknown""","""Yorkville West""","""Yellow Zone""",1.0,5626.316667,"""unknown""","""standard_rate""","""Myle Technologies Inc""",inf


In [99]:
# Zero column analysis -- Clean up output, i.e. pivot column names to rows with one column "count"
cols = [
    "passenger_count",
    "trip_distance",
    "duration_min",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
    "cbd_congestion_fee",
]
pl.concat(
    [df.select(pl.lit(c).alias("column"), (pl.col(c) == 0).sum().alias("zero_count")) for c in cols]
)

column,zero_count
str,u32
"""passenger_count""",24656
"""trip_distance""",90893
"""duration_min""",1927
"""fare_amount""",1398
"""extra""",1764424
"""mta_tax""",38170
"""tip_amount""",1118008
"""tolls_amount""",3259590
"""improvement_surcharge""",37694


In [100]:
# Tail analysis
cols = ["trip_distance", "passenger_count", "duration_min"]
cast_df = df.with_columns(pl.col(cols).cast(pl.Float64))
pl.concat(
    [
        cast_df.select(
            pl.lit(c).alias("column"),
            pl.col(c).min().alias("min"),
            pl.col(c).quantile(0.001).alias("p0.1"),
            pl.col(c).quantile(0.999).alias("p99.9"),
            pl.col(c).max().alias("max"),
        )
        for c in cols
    ]
)

column,min,p0.1,p99.9,max
str,f64,f64,f64,f64
"""trip_distance""",0.0,0.0,28.86,276423.57
"""passenger_count""",0.0,0.0,6.0,9.0
"""duration_min""",-51472.316667,0.066667,98.266667,5626.316667


In [101]:
# Hour-by-hour distribution
df.group_by(pl.col("pickup_datetime").dt.hour().alias("hour")).agg(
    pl.col("duration_min").mean().alias("avg_duration")
).sort("hour")

hour,avg_duration
i8,f64
0,13.785256
1,12.06919
2,12.763833
3,12.500513
4,14.155369
5,15.998068
6,15.67766
7,15.709326
8,15.574044


In [102]:
# Payment type distribution
df["payment_type"].value_counts()

payment_type,count
i64,u32
4,76481
5,1
2,390429
3,23773
0,540149
1,2444393


In [103]:
# Zone distribution
zone_counts = df["zone"].value_counts().sort("count", descending=True)
display(zone_counts)

zone,count
str,u32
"""Midtown Center""",169977
"""Upper East Side South""",163703
"""Upper East Side North""",155647
"""JFK Airport""",146137
"""Times Sq/Theatre District""",125829
"""Penn Station/Madison Sq West""",119131
"""Midtown East""",117930
"""Lincoln Square East""",110585
"""Upper West Side South""",96614


In [104]:
# Airport pickup distribution
df["is_airport_pickup"].value_counts()

is_airport_pickup,count
i8,u32
0,3239054
1,236172


In [105]:
# Borough pickup distribution
df["borough"].value_counts()

borough,count
str,u32
"""EWR""",377
"""Unknown""",8141
"""Manhattan""",3089275
"""Bronx""",14741
"""Staten Island""",256
"""Queens""",294986
"""N/A""",1380
"""Brooklyn""",66070


In [106]:
# N/A Borough breakout
df.filter(df["borough"] == "N/A")["zone"].value_counts()

zone,count
str,u32
"""Outside of NYC""",1380


In [107]:
df.filter((pl.col("fare_amount") == 0) & (pl.col("total_amount") == 0))

vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,borough,zone,service_zone,is_airport_pickup,duration_min,payment_type_str,rate_code_str,vendor_id_str,avg_speed_mph
i32,datetime[μs],datetime[μs],i64,f64,i64,i64,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,i8,f64,str,str,str,f64
2,2025-01-01 01:30:13,2025-01-01 01:30:31,1,0.0,1,0,226,226,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Queens""","""Sunnyside""","""Boro Zone""",0,0.3,"""cash""","""standard_rate""","""Curb Mobility, LLC""",0.0
1,2025-01-01 03:57:21,2025-01-01 04:02:26,1,0.4,1,0,158,125,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Manhattan""","""Meatpacking/West Village West""","""Yellow Zone""",0,5.083333,"""no_charge""","""standard_rate""","""Creative Mobile Technologies L…",4.721311
1,2025-01-01 03:27:25,2025-01-01 03:43:43,1,3.5,1,0,144,163,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Manhattan""","""Little Italy/NoLiTa""","""Yellow Zone""",0,16.3,"""dispute""","""standard_rate""","""Creative Mobile Technologies L…",12.883436
2,2025-01-01 04:03:25,2025-01-01 04:11:42,2,4.69,1,0,255,97,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Brooklyn""","""Williamsburg (North Side)""","""Boro Zone""",0,8.283333,"""cash""","""standard_rate""","""Curb Mobility, LLC""",33.971831
2,2025-01-01 04:24:55,2025-01-01 04:29:57,4,1.72,1,0,7,7,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Queens""","""Astoria""","""Boro Zone""",0,5.033333,"""cash""","""standard_rate""","""Curb Mobility, LLC""",20.503311
1,2025-01-02 01:48:11,2025-01-02 01:48:11,1,0.0,5,1,87,264,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Manhattan""","""Financial District North""","""Yellow Zone""",0,0.0,"""cash""","""negotiated_fare""","""Creative Mobile Technologies L…",NaN
1,2025-01-02 07:05:37,2025-01-02 07:05:59,1,0.0,1,0,20,20,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Bronx""","""Belmont""","""Boro Zone""",0,0.366667,"""no_charge""","""standard_rate""","""Creative Mobile Technologies L…",0.0
1,2025-01-02 07:39:50,2025-01-02 07:42:08,1,0.0,1,0,114,211,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Manhattan""","""Greenwich Village South""","""Yellow Zone""",0,2.3,"""dispute""","""standard_rate""","""Creative Mobile Technologies L…",0.0
1,2025-01-02 07:48:54,2025-01-02 07:49:06,1,0.0,1,0,79,79,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""Manhattan""","""East Village""","""Yellow Zone""",0,0.2,"""dispute""","""standard_rate""","""Creative Mobile Technologies L…",0.0


In [108]:
df.filter((pl.col("trip_distance") == 0) & (pl.col("duration_min") > 1))

vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,borough,zone,service_zone,is_airport_pickup,duration_min,payment_type_str,rate_code_str,vendor_id_str,avg_speed_mph
i32,datetime[μs],datetime[μs],i64,f64,i64,i64,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,i8,f64,str,str,str,f64
1,2025-01-01 00:27:40,2025-01-01 00:59:30,1,0.0,1,0,168,76,1,50.5,0.0,0.5,0.0,6.94,1.0,58.94,0.0,0.0,0.0,"""Bronx""","""Mott Haven/Port Morris""","""Boro Zone""",0,31.833333,"""credit_card""","""standard_rate""","""Creative Mobile Technologies L…",0.0
1,2025-01-01 00:43:22,2025-01-01 00:46:18,1,0.0,5,0,48,48,1,5.0,0.0,0.0,3.0,0.0,1.0,9.0,0.0,0.0,0.0,"""Manhattan""","""Clinton East""","""Yellow Zone""",0,2.933333,"""credit_card""","""negotiated_fare""","""Creative Mobile Technologies L…",0.0
1,2025-01-01 00:14:57,2025-01-01 00:17:44,1,0.0,1,0,79,79,3,3.0,3.5,0.5,0.0,0.0,1.0,8.0,2.5,0.0,0.0,"""Manhattan""","""East Village""","""Yellow Zone""",0,2.783333,"""no_charge""","""standard_rate""","""Creative Mobile Technologies L…",0.0
1,2025-01-01 00:34:58,2025-01-01 01:09:03,1,0.0,99,0,78,139,1,53.5,0.0,0.5,0.0,6.94,1.0,61.94,0.0,0.0,0.0,"""Bronx""","""East Tremont""","""Boro Zone""",0,34.083333,"""credit_card""","""null_or_unknown""","""Creative Mobile Technologies L…",0.0
1,2025-01-01 00:22:03,2025-01-01 00:24:00,1,0.0,1,0,237,237,1,3.7,3.5,0.5,1.7,0.0,1.0,10.4,2.5,0.0,0.0,"""Manhattan""","""Upper East Side South""","""Yellow Zone""",0,1.95,"""credit_card""","""standard_rate""","""Creative Mobile Technologies L…",0.0
1,2025-01-01 00:37:13,2025-01-01 01:26:20,1,0.0,99,0,49,247,1,40.5,0.0,0.5,0.0,0.0,1.0,42.0,0.0,0.0,0.0,"""Brooklyn""","""Clinton Hill""","""Boro Zone""",0,49.116667,"""credit_card""","""null_or_unknown""","""Creative Mobile Technologies L…",0.0
2,2025-01-01 00:13:46,2025-01-01 00:29:45,2,0.0,1,0,234,231,1,13.5,1.0,0.5,3.7,0.0,1.0,22.2,2.5,0.0,0.0,"""Manhattan""","""Union Sq""","""Yellow Zone""",0,15.983333,"""credit_card""","""standard_rate""","""Curb Mobility, LLC""",0.0
2,2025-01-01 00:32:18,2025-01-01 00:39:23,2,0.0,1,0,231,158,1,7.2,1.0,0.5,1.0,0.0,1.0,13.2,2.5,0.0,0.0,"""Manhattan""","""TriBeCa/Civic Center""","""Yellow Zone""",0,7.083333,"""credit_card""","""standard_rate""","""Curb Mobility, LLC""",0.0
2,2025-01-01 00:45:42,2025-01-01 01:23:00,3,0.0,1,0,158,262,1,28.9,1.0,0.5,3.0,0.0,1.0,36.9,2.5,0.0,0.0,"""Manhattan""","""Meatpacking/West Village West""","""Yellow Zone""",0,37.3,"""credit_card""","""standard_rate""","""Curb Mobility, LLC""",0.0


In [ ]:
# Cleaning Calls!

keep_list = [
    "vendor_id",
    "pickup_datetime",
    "passenger_count",
    "store_and_fwd_flag",
    "pickup_location_id",
    "borough",
    "is_airport_pickup",
    "trip_distance",
    "duration_min",
    "avg_speed_mph",
]
df = df.drop_nulls(subset=keep_list)
df = df.filter(
    pl.col("duration_min") > 1,
    pl.col("trip_distance") > 0,
    pl.col("pickup_datetime").is_between(pl.date(2025, 1, 5), pl.date(2025, 1, 31)),
    pl.col("total_amount") > 0,
    pl.col("passenger_count") <= 6.0,
    pl.col("avg_speed_mph").is_between(1, 85),
)

df.select(keep_list)


vendor_id,pickup_datetime,passenger_count,store_and_fwd_flag,pickup_location_id,borough,is_airport_pickup,trip_distance,duration_min,avg_speed_mph
i32,datetime[μs],i64,i64,i32,str,i8,f64,f64,f64
2,2025-01-05 00:00:02,1,0,138,"""Queens""",1,8.66,17.75,29.273239
2,2025-01-05 00:24:51,1,0,138,"""Queens""",1,9.73,25.6,22.804688
2,2025-01-05 00:54:12,1,0,138,"""Queens""",1,10.33,23.766667,26.078541
1,2025-01-05 00:27:27,0,0,138,"""Queens""",1,10.3,16.633333,37.154309
2,2025-01-05 00:10:19,1,0,239,"""Manhattan""",0,1.71,5.733333,17.895349
2,2025-01-05 00:29:32,1,0,236,"""Manhattan""",0,1.8,7.65,14.117647
2,2025-01-05 00:06:01,1,0,48,"""Manhattan""",0,3.05,14.45,12.66436
2,2025-01-05 00:48:01,1,0,142,"""Manhattan""",0,5.81,15.816667,22.040042
1,2025-01-05 00:22:44,1,0,138,"""Queens""",1,9.0,20.733333,26.045016


In [110]:
# one-hot encode boroughs column
clean_df = df.to_dummies("borough")
clean_df.columns = [col.lower() for col in clean_df.columns]
del df
display(clean_df)

vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee,borough_bronx,borough_brooklyn,borough_ewr,borough_manhattan,borough_n/a,borough_queens,borough_staten island,borough_unknown,zone,service_zone,is_airport_pickup,duration_min,payment_type_str,rate_code_str,vendor_id_str,avg_speed_mph
i32,datetime[μs],datetime[μs],i64,f64,i64,i64,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,u8,u8,u8,u8,u8,u8,u8,u8,str,str,i8,f64,str,str,str,f64
2,2025-01-05 00:00:02,2025-01-05 00:17:47,1,8.66,1,0,138,43,1,35.2,6.0,0.5,12.85,6.94,1.0,64.24,0.0,1.75,0.0,0,0,0,0,0,1,0,0,"""LaGuardia Airport""","""Airports""",1,17.75,"""credit_card""","""standard_rate""","""Curb Mobility, LLC""",29.273239
2,2025-01-05 00:24:51,2025-01-05 00:50:27,1,9.73,1,0,138,61,1,40.8,6.0,0.5,10.01,0.0,1.0,60.06,0.0,1.75,0.0,0,0,0,0,0,1,0,0,"""LaGuardia Airport""","""Airports""",1,25.6,"""credit_card""","""standard_rate""","""Curb Mobility, LLC""",22.804688
2,2025-01-05 00:54:12,2025-01-05 01:17:58,1,10.33,1,0,138,62,1,41.5,6.0,0.5,10.0,0.0,1.0,60.75,0.0,1.75,0.0,0,0,0,0,0,1,0,0,"""LaGuardia Airport""","""Airports""",1,23.766667,"""credit_card""","""standard_rate""","""Curb Mobility, LLC""",26.078541
1,2025-01-05 00:27:27,2025-01-05 00:44:05,0,10.3,1,0,138,162,2,39.4,10.25,0.5,0.0,6.94,1.0,58.09,2.5,1.75,0.0,0,0,0,0,0,1,0,0,"""LaGuardia Airport""","""Airports""",1,16.633333,"""cash""","""standard_rate""","""Creative Mobile Technologies L…",37.154309
2,2025-01-05 00:10:19,2025-01-05 00:16:03,1,1.71,1,0,239,24,1,9.3,1.0,0.5,2.86,0.0,1.0,17.16,2.5,0.0,0.0,0,0,0,1,0,0,0,0,"""Upper West Side South""","""Yellow Zone""",0,5.733333,"""credit_card""","""standard_rate""","""Curb Mobility, LLC""",17.895349
2,2025-01-05 00:29:32,2025-01-05 00:37:11,1,1.8,1,0,236,143,1,10.7,1.0,0.5,3.14,0.0,1.0,18.84,2.5,0.0,0.0,0,0,0,1,0,0,0,0,"""Upper East Side North""","""Yellow Zone""",0,7.65,"""credit_card""","""standard_rate""","""Curb Mobility, LLC""",14.117647
2,2025-01-05 00:06:01,2025-01-05 00:20:28,1,3.05,1,0,48,140,1,17.0,1.0,0.5,5.69,0.0,1.0,28.44,2.5,0.0,0.75,0,0,0,1,0,0,0,0,"""Clinton East""","""Yellow Zone""",0,14.45,"""credit_card""","""standard_rate""","""Curb Mobility, LLC""",12.66436
2,2025-01-05 00:48:01,2025-01-05 01:03:50,1,5.81,1,0,142,232,1,26.1,1.0,0.5,6.37,0.0,1.0,38.22,2.5,0.0,0.75,0,0,0,1,0,0,0,0,"""Lincoln Square East""","""Yellow Zone""",0,15.816667,"""credit_card""","""standard_rate""","""Curb Mobility, LLC""",22.040042
1,2025-01-05 00:22:44,2025-01-05 00:43:28,1,9.0,1,0,138,239,2,36.6,10.25,0.5,0.0,6.94,1.0,55.29,2.5,1.75,0.0,0,0,0,0,0,1,0,0,"""LaGuardia Airport""","""Airports""",1,20.733333,"""cash""","""standard_rate""","""Creative Mobile Technologies L…",26.045016


# EDA Notes

### Fields Known at Pickup:

- **VendorID** - I assume this has to be the taxi proprietor
- RatecodeID
- **tpep_pickup_datetime** - goes without saying
- **passenger_count** - again, you should know how many people are getting in the car
- **store_and_fwd_flag**(?) - not sure about this one
- **PULocationID** - again, you should know where you are
- **is_airport_ride** - Derived from PULocationID

### Pickup/Dropoff

- Earliest pickup datetime is after the earliest dropoff datetime
  - Going to need to remove records where:
    - tpep_dropoff_datetime <= tpep_pickup_datetime

### Zero Values

- **Trip Distance:** 90,893 zero distance records
- **Passenger Count:** 24,656 zero passenger records
- **Fare Amount:** 1,398 zero fare amount records (May or may not be legitimate)
  - Small population suggests that this is an error rather than legit zeros.
- **Total Amount:** 559 zero total amount records (May or may not be legitimate)
  - Small population suggests that this is an error rather than legit zeros.
  - _NOTE:_ Zero total amount population is smaller than zero fare amount population./nWarrants further investigation as this could be either:
    - The ride was actually free and the individual provided a tip or there was a categorical surcharge present
    - 559 of the 1,398 zero fare amount records should have a larger value present and the remainder were legitimate zero fare rides
- **Trip Duration:** Only 1,927 zero minute duration rides is vast difference from the 90,893 zero trip distance records
  - Could be cab starts meter after stopping for pickup and then ride is cancelled with time charged?
    - I don't live in NY, so I'm not sure how this works.

### Tail Analysis

- Trip distance has massive max error tail
  - p99.9 is only 28.86, so 276423.57 is both, per the dataset and mathematically, impossible
- Passenger count has only 18 records > 6.0 (four 7.0 records, eleven 8.0 records, and three 9.0 records)
  - Given there are millions of records, it seems wiser to just cut these out, although it is technically possible that there may have been babies present with parents or people sitting on laps illegally, just saying.
- Any negative trip duration/trip distance values can be removed from the dataset.

### Cross-Column Relationship Issues

- **Trip Duration x Trip Distance**
  - Potential problem in terms of its error tail. It's hard to say where the errors are and where some of them are just truly long trips when cross referenced with **Trip Distance.** There is a 91 mile trip where the meter ran for 5000+ minutes. Given that that is 3.5 days of usage, I doubt the time is entirely legitimate, but it is a long haul trip for sure. There are also hundreds where it ran for 300+ minutes on a 15-20 mile trip. So it's difficult to pin down where to actually remove meter errors and which are legitimate fares/trip durations.
  - Duration should have a minimum cutoff as well in relation to distance of some sort. Trips less than 1 minute seem ridiculous logically, but also, if something has a long distance then there needs to be a logical amount of trip duration for that distance.
  - _NOTE:_ Perhaps get trimmed, weighted average for time per mile travelled to determine a reasonable min/max for the cross-column relationship and remove anything that falls outside of that. I assume there are going to be accuracy issues as part of this. Could also create some sort of likelihood value of the mile/minute value and then filter based on that?
-
